In [1]:
from docling.datamodel.base_models import InputFormat
from docling.datamodel.pipeline_options import PdfPipelineOptions, EasyOcrOptions, PipelineOptions
from docling.datamodel.pipeline_options import AcceleratorOptions, AcceleratorDevice
from docling.document_converter import DocumentConverter, PdfFormatOption, ImageFormatOption
from docling_core.types.doc import ImageRefMode, PictureItem, TableItem, TextItem

from pathlib import Path

from googletrans import Translator
import tqdm

In [2]:
output_dir = Path("extractions")
output_dir.mkdir(parents=True, exist_ok=True)

In [3]:
cpu_accelerator_options = AcceleratorOptions(
    num_threads=8, device=AcceleratorDevice.CPU
)

gpu_accelerator_options = AcceleratorOptions(
    device=AcceleratorDevice.CUDA
)

In [5]:
IMAGE_RESOLUTION_SCALE = 2.0

digital_pipeline_options = PdfPipelineOptions(
    images_scale = IMAGE_RESOLUTION_SCALE,
    generate_picture_images = True,
    generate_table_images = True,
    accelerator_options = cpu_accelerator_options,
    do_ocr=False
)

ocr_options = EasyOcrOptions(lang=["ch_tra"], force_full_page_ocr=True)

ocr_pipeline_options = PdfPipelineOptions(
    images_scale = IMAGE_RESOLUTION_SCALE,
    generate_picture_images = True,
    generate_table_images = True,
    accelerator_options = cpu_accelerator_options,
    ocr_options = ocr_options
)

image_pipeline_options = PipelineOptions(
    accelerator_options=cpu_accelerator_options,
    ocr_options=ocr_options
)

converter = DocumentConverter(
    format_options={
        InputFormat.PDF: PdfFormatOption(pipeline_options=ocr_pipeline_options),
        InputFormat.IMAGE: ImageFormatOption(pipeline_options=ocr_pipeline_options)
    }
)

converter.initialize_pipeline(InputFormat.PDF)
converter.initialize_pipeline(InputFormat.IMAGE)



In [6]:
file_name = "Peer-Review-0124-Dansk-Konsensus-ACP-DOI.pdf"
# file_name = "Rapport.pdf"
file_name = "chinese.pdf"

source = "./docs/" + file_name  
result = converter.convert(source)
doc_filename = result.input.file.stem

In [20]:
html_filename = output_dir / f"{doc_filename}.html"
result.document.save_as_html(html_filename, image_mode=ImageRefMode.EMBEDDED)

In [12]:
do_translate = True

if do_translate:
    translator = Translator()
    for text in tqdm.tqdm(result.document.texts):
        translation =  await translator.translate(text.text, dest="en")
        text.text = translation.text

100%|██████████| 74/74 [00:03<00:00, 18.64it/s]


In [14]:
html_filename = output_dir / f"{doc_filename}-translated.html"
result.document.save_as_html(html_filename, image_mode=ImageRefMode.EMBEDDED)

